# SwAV ON MNIST — 100 LABELS — SAAD

## 0. COLAB SETUP

In [1]:
import os, sys, subprocess

REPO = "saadrhouddal/Deep-learning-swav-mnist-"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = "/content/Deep-learning-swav-mnist-"
    if not os.path.exists(REPO_DIR):
        # Private repo: add a GITHUB_TOKEN secret (key icon in the left sidebar)
        try:
            from google.colab import userdata
            url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
        except Exception:
            url = f"https://github.com/{REPO}.git"   # public repo
        subprocess.run(["git", "clone", "-q", url, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    # Local run: notebook is in notebooks/, repo root is one level up
    REPO_DIR = os.path.abspath("..")

sys.path.insert(0, REPO_DIR)
DATA_DIR = os.path.join(REPO_DIR, "data")
CKPT_DIR = os.path.join(REPO_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
print("Colab:", IN_COLAB, "| Repo:", REPO_DIR)

Colab: True | Repo: /content/Deep-learning-swav-mnist-


In [ ]:
# Optional: save checkpoints on Google Drive (Colab files are deleted at the end of the session)
# from google.colab import drive
# drive.mount("/content/drive")
# CKPT_DIR = "/content/drive/MyDrive/swav_mnist_checkpoints"
# os.makedirs(CKPT_DIR, exist_ok=True)

## 1. IMPORTS

In [2]:
import time, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import torchvision
from torchvision import transforms

from preprocessing import (
    get_data, set_seed, get_device, get_loaders, get_unlabeled_loader,
    split_checksum, NORMALIZE, MNIST_MEAN, MNIST_STD, SEED,
)

set_seed(SEED)
device = get_device()
print("torch", torch.__version__, "| device:", device)

torch 2.14.1+cu130 | device: cuda


## 2. DATA LOADING

In [3]:
data = get_data(data_dir=DATA_DIR)

100%|██████████| 9.91M/9.91M [00:00<00:00, 14.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 402kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.78MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.0MB/s]


[preprocessing] Created split file: /content/Deep-learning-swav-mnist-/labeled_indices_seed42.json


## 3. SHARED DATASETS

In [4]:
# ---- Self-supervised pretraining (SwAV) — NO labels ----
X_unlabeled = data["X_unlabeled"]   # (60000, 1, 28, 28) normalized — all train images
X_train_raw = data["X_train_raw"]   # (60000, 28, 28) uint8 — for augmentation / multi-crop pipelines

# ---- The 100 labeled images (10 per digit, same split for everyone) ----
X_lab = data["X_lab"]               # (100, 1, 28, 28) normalized
y_lab = data["y_lab"]               # (100,)
labeled_idx = data["labeled_idx"]   # indices of the 100 images in the train set

# ---- Final evaluation ----
X_test = data["X_test"]             # (10000, 1, 28, 28) normalized
y_test = data["y_test"]             # (10000,)
X_test_raw = data["X_test_raw"]     # (10000, 28, 28) uint8 — for visualization

# ---- Analysis ONLY (t-SNE colors, clustering metrics) — never for training ----
y_train = data["y_train"]           # (60000,)

print("X_unlabeled:", tuple(X_unlabeled.shape))
print("X_lab:", tuple(X_lab.shape), "| per class:", torch.bincount(y_lab, minlength=10).tolist())
print("X_test:", tuple(X_test.shape))
print("Split checksum:", split_checksum(labeled_idx))

X_unlabeled: (60000, 1, 28, 28)
X_lab: (100, 1, 28, 28) | per class: [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
X_test: (10000, 1, 28, 28)
Split checksum: 3521659ad9


## 4. DATALOADERS

In [5]:
# Labeled (100 images) and test loaders
lab_loader, test_loader = get_loaders(data, batch_size_lab=100, batch_size_test=1000)

# Unlabeled loader for SwAV: create it later with the multi-crop transform
# unlabeled_loader = get_unlabeled_loader(data, transform=multicrop_transform, batch_size=256)

xb, yb = next(iter(lab_loader))
print("lab batch:", tuple(xb.shape), "| test batches:", len(test_loader))

lab batch: (100, 1, 28, 28) | test batches: 10
